# Lab 12 - K Means Clustering: Pollution Profiles

This notebook applies **Lab 12 - K Means Clustering** to group city-hour pollution profiles without using the supervised target during fitting.


## Lab 12 concepts used

- Select numeric clustering features.
- Scale features before K-Means.
- Use inertia and silhouette score to compare cluster counts.
- Profile clusters using original feature averages and hazardous-event rates.
- Visualize clusters with PCA.

`Hazardous_Event` is not used to create clusters; it is only used after fitting to interpret cluster risk.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

cluster_features = [
    'PM10_ug_m3', 'PM2_5_ug_m3', 'Carbon_Monoxide_ug_m3',
    'Nitrogen_Dioxide_ug_m3', 'Ozone_ug_m3', 'Dust_ug_m3',
    'UV_Index', 'hour', 'dayofweek'
]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model_df = latest_rows(add_time_features(data), 50000)
X = model_df[cluster_features]
cluster_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_scaled = cluster_preprocess.fit_transform(X)


In [ ]:
cluster_results = []
for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    sample_size = min(10000, len(X_scaled))
    sample_idx = np.random.default_rng(42).choice(len(X_scaled), size=sample_size, replace=False)
    cluster_results.append({
        'k': k,
        'inertia': kmeans.inertia_,
        'silhouette_sample': silhouette_score(X_scaled[sample_idx], labels[sample_idx]),
    })
cluster_results_df = pd.DataFrame(cluster_results)
cluster_results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cluster_results_df['k'], cluster_results_df['inertia'], marker='o')
axes[0].set_title('K-Means elbow curve')
axes[0].set_xlabel('Number of clusters')
axes[0].set_ylabel('Inertia')
axes[1].plot(cluster_results_df['k'], cluster_results_df['silhouette_sample'], marker='o', color='darkgreen')
axes[1].set_title('Silhouette score sample')
axes[1].set_xlabel('Number of clusters')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.show()


In [ ]:
selected_k = 4
kmeans = KMeans(n_clusters=selected_k, n_init=20, random_state=42)
model_df['cluster'] = kmeans.fit_predict(X_scaled)
profile_cols = cluster_features + ['European_AQI', 'Hazardous_Event']
cluster_profile = model_df.groupby('cluster')[profile_cols].mean().round(2)
cluster_profile['rows'] = model_df['cluster'].value_counts().sort_index()
cluster_profile


In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
plot_df = model_df[['cluster', 'City', 'Hazardous_Event']].copy()
plot_df['PC1'] = coords[:, 0]
plot_df['PC2'] = coords[:, 1]
plot_df = plot_df.sample(n=min(8000, len(plot_df)), random_state=42)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='cluster', palette='tab10', alpha=0.45, linewidth=0)
plt.title('K-Means pollution-profile clusters')
plt.show()


## What was learned from Lab 12

K-Means gives an unsupervised view of recurring pollution profiles. After fitting, the clusters can be interpreted by pollutant levels, time patterns, city composition, AQI, and hazardous-event rates to support recommendations.
